In [ ]:
import torch
import torch.nn as nn
import math

class ReconciledPINN(nn.Module):
    """
    Standard MLP for PINN tasks.
    Spec: 4 hidden layers x 64 units, Tanh activation[cite: 2].
    """
    def __init__(self, in_dim=2, out_dim=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, out_dim)
        )

    def forward(self, *inputs):
        combined = torch.cat(inputs, dim=1)
        return self.net(combined)

def compute_physics_loss(model, x_f, t_f, alpha):
    x_f.requires_grad = True
    t_f.requires_grad = True
    u = model(x_f, t_f)
    u_t = torch.autograd.grad(u, t_f, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_x = torch.autograd.grad(u, x_f, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x_f, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
    residual = u_t - alpha * u_xx
    return torch.mean(residual**2)

def create_heat_pinn_factory(alpha=0.01, epochs=500, n_points=2000):
    def pinn_solver_factory(k):
        ic_function = lambda x: torch.sin(k * math.pi * x)
        model = ReconciledPINN(in_dim=2, out_dim=1)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        
        x_f = torch.rand((n_points, 1))
        t_f = torch.rand((n_points, 1))
        x_ic = torch.rand((n_points // 4, 1))
        t_ic = torch.zeros((n_points // 4, 1))
        u_ic_exact = ic_function(x_ic) 
        t_bc = torch.rand((n_points // 4, 1))
        x_bc_0 = torch.zeros((n_points // 4, 1))
        x_bc_1 = torch.ones((n_points // 4, 1))
        
        for epoch in range(epochs):
            optimizer.zero_grad()
            loss_res = compute_physics_loss(model, x_f, t_f, alpha)
            u_ic_pred = model(x_ic, t_ic)
            loss_ic = torch.mean((u_ic_pred - u_ic_exact)**2)
            u_bc_0_pred = model(x_bc_0, t_bc)
            u_bc_1_pred = model(x_bc_1, t_bc)
            loss_bc = torch.mean(u_bc_0_pred**2) + torch.mean(u_bc_1_pred**2)
            
            total_loss = loss_res + loss_ic + loss_bc
            total_loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            x_test = torch.linspace(0, 1, 100).view(-1, 1)
            t_test = torch.ones((100, 1)) * 0.01
            u_pred = model(x_test, t_test).cpu()
            
        loss_dict = {'res_loss': loss_res.item(), 'ic_loss': loss_ic.item(), 'bc_loss': loss_bc.item()}
        return u_pred, loss_dict
    return pinn_solver_factory